In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10) # 10 classes for MNIST
])

print("Custom CNN model created successfully!")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Custom CNN model created successfully!


## We will use the MNIST dataset. Since our custom CNN is designed for 28x28 grayscale images, we will only normalize the pixel values.

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, datasets
import cv2

# Load and preprocess MNIST dataset
(train_images, train_labels), (test_images, test_labels) = datasets.mnist.load_data()

# Reshape to add channel dimension for CNN (28, 28) -> (28, 28, 1)
train_images = train_images.reshape(train_images.shape[0], 28, 28, 1)
test_images = test_images.reshape(test_images.shape[0], 28, 28, 1)

# Normalize pixel values to be between 0 and 1
train_images, test_images = train_images / 255.0, test_images / 255.0

# Class names for MNIST (digits 0-9)
class_names = [str(i) for i in range(10)]

print(f"Training data shape: {train_images.shape}")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Training data shape: (60000, 28, 28, 1)


In [5]:
num_train_samples = 5000 # Use a smaller subset for training
num_val_samples = 1000   # Use a smaller subset for validation

# Create a training subset
train_images_subset = train_images[:num_train_samples]
train_labels_subset = train_labels[:num_train_samples]

# Create a validation subset from the remaining training data
val_images = train_images[num_train_samples:num_train_samples + num_val_samples]
val_labels = train_labels[num_train_samples:num_train_samples + num_val_samples]

print(f"Training subset shape: {train_images_subset.shape}")
print(f"Validation subset shape: {val_images.shape}")

Training subset shape: (5000, 28, 28, 1)
Validation subset shape: (1000, 28, 28, 1)


# Compile and Train the Custom CNN


In [6]:
EPOCHS = 5 # Reduced epochs for a quicker demonstration

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

history = model.fit(
    train_images_subset, train_labels_subset,
    epochs=EPOCHS,
    validation_data=(val_images, val_labels)
)

print("Training complete!")

Epoch 1/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 10s 32ms/step - accuracy: 0.8140 - loss: 0.6292 - val_accuracy: 0.9280 - val_loss: 0.2208
Epoch 2/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9510 - loss: 0.1663 - val_accuracy: 0.9240 - val_loss: 0.2373
Epoch 3/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9604 - loss: 0.1183 - val_accuracy: 0.9730 - val_loss: 0.1055
Epoch 4/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9776 - loss: 0.0751 - val_accuracy: 0.9680 - val_loss: 0.1028
Epoch 5/5
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9800 - loss: 0.0645 - val_accuracy: 0.9630 - val_loss: 0.1198
Training complete!


### Evaluate the Original Model's Performance

evaluate the trained custom CNN on the test dataset to get its baseline accuracy and prepare for inference speed measurement.

In [8]:
loss, accuracy = model.evaluate(test_images, test_labels, verbose=2)
print(f"Original Model Accuracy: {accuracy*100:.2f}%")

# Save the original (float32) model to measure its size correctly
original_model_filepath = 'saved_original_model.keras'
model.save(original_model_filepath)
print("Original (float32) model saved successfully!")

313/313 - 1s - 2ms/step - accuracy: 0.9679 - loss: 0.0919
Original Model Accuracy: 96.79%
Original (float32) model saved successfully!


### Measure Original Model Inference Speed

To measure inference speed, we'll run predictions on a batch of test images and calculate the average time per image.

In [16]:
import time
import numpy as np

# Use a subset of test images for speed measurement
num_inference_samples = 100
inference_images = test_images[:num_inference_samples]

# Warm-up runs
for _ in range(10):
    _ = model.predict(inference_images, verbose=0)

# Measure inference time over multiple runs
num_runs = 10
times = []
for _ in range(num_runs):
    start_time = time.time()
    _ = model.predict(inference_images, verbose=0)
    end_time = time.time()
    times.append(end_time - start_time)

avg_inference_time_per_batch = np.mean(times)
avg_inference_time_ms_per_image = (avg_inference_time_per_batch / num_inference_samples) * 1000

print(f"Average inference time (Original Model): {avg_inference_time_ms_per_image:.2f} ms per image")

original_model_metrics = {
    'accuracy': accuracy,
    'inference_time_ms_per_image': avg_inference_time_ms_per_image
}
print(f"Original model metrics: {original_model_metrics}")

Average inference time (Original Model): 0.77 ms per image
Original model metrics: {'accuracy': 0.9678999781608582, 'inference_time_ms_per_image': np.float64(0.7715361118316652)}


### Quantize the Model

 quantize the `model` to INT8 using TensorFlow Lite's Post-Training Quantization (PTQ) technique. We will use dynamic range quantization, which quantizes weights and performs activations dynamically. For full integer quantization, a representative dataset would be needed, but dynamic range quantization is a good starting point for this demonstration.

In [12]:
import os

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

try:
    quantized_tflite_model = converter.convert()
    # Save the quantized model
    with open('quantized_model.tflite', 'wb') as f:
        f.write(quantized_tflite_model)
    print("Model quantized successfully and saved as 'quantized_model.tflite'!")

    # Measure the size of the original and quantized models
    original_model_size = os.path.getsize(original_model_filepath) / (1024 * 1024) # in MB
    quantized_model_size = os.path.getsize('quantized_model.tflite') / (1024 * 1024) # in MB

    print(f"Original model size: {original_model_size:.2f} MB")
    print(f"Quantized model size: {quantized_model_size:.2f} MB")

except Exception as e:
    print(f"Error during quantization or saving: {e}")
    # Set default values or raise to prevent further errors if critical
    original_model_size = 0.0
    quantized_model_size = 0.0
    print("Quantization failed. Setting model sizes to 0.0.")


Saved artifact at '/tmp/tmptf4yy74x'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  135459967236816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135459967237584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135459967236624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135459967237008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135459967237392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135459967235280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135459967235472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135459967237200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135459960915344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135459960914576: TensorSpec(shape=(), dtype=tf.resource, name=None)
Model quantized s

### Load and Evaluate the Quantized Model

In [ ]:
# Load the TFLite model and allocate tensors
interpreter = tf.lite.Interpreter(model_path='quantized_model.tflite')
interpreter.allocate_tensors()

# Get input and output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Convert test images to the input type required by the TFLite model
input_shape = input_details[0]['shape']
input_dtype = input_details[0]['dtype']


if input_dtype == tf.int8:
    # For full integer quantization, input data often needs to be scaled to int8.
    # However, dynamic range quantization often keeps float32 input.
    # We are stick with float32 for dynamic range as the initial approach
    quantized_test_images = test_images.astype(np.float32)
else:
    quantized_test_images = test_images.astype(input_dtype)

# Function to run inference for a single image
def run_tflite_inference(interpreter, input_data):
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])
    return output_data

# Evaluate accuracy of the quantized model
quantized_predictions = []
# Ensure test_labels is a 1D array for comparison
flat_test_labels = test_labels.flatten()
for i in range(len(test_images)):
    input_data = np.expand_dims(quantized_test_images[i], axis=0)
    output_data = run_tflite_inference(interpreter, input_data)
    quantized_predictions.append(np.argmax(output_data))

quantized_accuracy = np.mean(quantized_predictions == flat_test_labels) # Use flat_test_labels here
print(f"Quantized Model Accuracy: {quantized_accuracy*100:.2f}%")

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Quantized Model Accuracy: 96.78%


### Measure Quantized Model Inference Speed

In [ ]:
quantized_inference_images = quantized_test_images[:num_inference_samples]

# Warm-up runs for quantized model
for _ in range(10):
    _ = run_tflite_inference(interpreter, np.expand_dims(quantized_inference_images[0], axis=0)) # Use first image for warmup

# Measure inference time over multiple runs
quantized_times = []
for _ in range(num_runs):
    batch_start_time = time.time()
    for i in range(num_inference_samples):
        input_data = np.expand_dims(quantized_inference_images[i], axis=0)
        _ = run_tflite_inference(interpreter, input_data)
    batch_end_time = time.time()
    quantized_times.append(batch_end_time - batch_start_time)

avg_quantized_inference_time_per_batch = np.mean(quantized_times)
avg_quantized_inference_time_ms_per_image = (avg_quantized_inference_time_per_batch / num_inference_samples) * 1000

print(f"Average inference time (Quantized Model): {avg_quantized_inference_time_ms_per_image:.2f} ms per image")

quantized_model_metrics = {
    'accuracy': quantized_accuracy,
    'inference_time_ms_per_image': avg_quantized_inference_time_ms_per_image
}

Average inference time (Quantized Model): 0.11 ms per image


### Compare Original vs. Quantized Model

In [ ]:
print("\n--- Model Comparison ---")
print(f"Original Model Size: {original_model_size:.2f} MB")
print(f"Quantized Model Size: {quantized_model_size:.2f} MB")

# Handle potential division by zero if original_model_size is 0
if original_model_size > 0:
    print(f"Size Reduction: {((original_model_size - quantized_model_size) / original_model_size) * 100:.2f}%")
else:
    print("Size Reduction: N/A (Original model size is 0)")

print(f"\nOriginal Model Accuracy: {original_model_metrics['accuracy']*100:.2f}%")
print(f"Quantized Model Accuracy: {quantized_model_metrics['accuracy']*100:.2f}%")
print(f"Accuracy Difference: {(original_model_metrics['accuracy'] - quantized_model_metrics['accuracy'])*100:.2f}%")

print(f"\nOriginal Model Inference Time: {original_model_metrics['inference_time_ms_per_image']:.2f} ms/image")
print(f"Quantized Model Inference Time: {quantized_model_metrics['inference_time_ms_per_image']:.2f} ms/image")

if quantized_model_metrics['inference_time_ms_per_image'] > 0:
    print(f"Inference Speedup: {original_model_metrics['inference_time_ms_per_image'] / quantized_model_metrics['inference_time_ms_per_image']:.2f}x")
else:
    print("Inference Speedup: N/A (Quantized model inference time is 0)")

print("\nThis comparison highlights the trade-offs: quantization significantly reduces model size and often improves inference speed, potentially with a small reduction in accuracy.")


--- Model Comparison ---
Original Model Size: 1.11 MB
Quantized Model Size: 0.10 MB
Size Reduction: 91.11%

Original Model Accuracy: 96.79%
Quantized Model Accuracy: 96.78%
Accuracy Difference: 0.01%

Original Model Inference Time: 0.77 ms/image
Quantized Model Inference Time: 0.11 ms/image
Inference Speedup: 6.97x

This comparison highlights the trade-offs: quantization significantly reduces model size and often improves inference speed, potentially with a small reduction in accuracy.
